In [1]:
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)


In [ ]:
DATA_PATH = "merged_df_cleaned.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

if "survived" not in df.columns:
    raise ValueError("Expected target column 'survived' was not found.")

df = df.dropna(subset=["survived"]).copy()
df["survived"] = df["survived"].astype(int)

X_full = df.drop(columns=["survived"])
y = df["survived"]

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

print("Dataset shape:", df.shape)
print("Train shape:", X_train_full.shape)
print("Test shape:", X_test_full.shape)
print("Target distribution:\n", y.value_counts(normalize=True).rename("ratio"))


Dataset shape: (125922, 38)
Train shape: (100737, 37)
Test shape: (25185, 37)
Target distribution:
 survived
1    0.794555
0    0.205445
Name: ratio, dtype: float64


In [ ]:
# Grouping features together to make feature selection a little bit quicker
genre_flag_cols = [c for c in X_train_full.columns if c.startswith("genre_")]

feature_groups = {
    "numeric_core": ["nearest_distance", "rating_val", "review_cnt", "save_cnt"],
    "budget": ["budget_dinner", "budget_lunch", "budget_dinner_num", "budget_lunch_num"],
    "location": ["nearest_station", "municipalities_1", "municipalities_2"],
    "holiday": ["holiday"],
    "genre_text": ["genre"],
    "genre_flags": genre_flag_cols,
}

for group_name, cols in feature_groups.items():
    print(f"{group_name}: {len(cols)} columns")


numeric_core: 4 columns
budget: 4 columns
location: 3 columns
holiday: 1 columns
genre_text: 1 columns
genre_flags: 20 columns


In [ ]:
#Setting up preprocessor and 5-fold cross validation

def make_preprocessor(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numeric_cols = X.select_dtypes(exclude=["object", "category", "string"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=30)),
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ], remainder="drop")


def evaluate_feature_set(selected_columns, cv_splits):
    X_subset = X_train_full[selected_columns].copy()
    preprocessor = make_preprocessor(X_subset)

    baseline_model = Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            solver="liblinear",
            C=0.2,
            random_state=SEED,
        )),
    ])

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=SEED)
    scores = cross_val_score(baseline_model, X_subset, y_train, cv=cv, scoring="f1_macro", n_jobs=1)
    return float(scores.mean()), float(scores.std())


In [ ]:
#Trying to identify optimal feature groups - ended up with essentially everything included
feature_rows = []
group_names = list(feature_groups.keys())

for combo_size in range(1, len(group_names) + 1):
    for combo in itertools.combinations(group_names, combo_size):
        selected_columns = []
        for group_name in combo:
            selected_columns.extend(feature_groups[group_name])

        cv3_mean, cv3_std = evaluate_feature_set(selected_columns, cv_splits=3)
        feature_rows.append({
            "feature_set": " + ".join(combo),
            "group_names": combo,
            "n_columns": len(selected_columns),
            "cv3_mean": cv3_mean,
            "cv3_std": cv3_std,
        })

feature_search_df = pd.DataFrame(feature_rows).sort_values("cv3_mean", ascending=False)
print("Total combinations tested:", len(feature_search_df))
print(feature_search_df.head(10)[["feature_set", "n_columns", "cv3_mean"]])

top_candidates = feature_search_df.head(12).copy()
refined_rows = []

for _, row in top_candidates.iterrows():
    selected_columns = []
    for group_name in row["group_names"]:
        selected_columns.extend(feature_groups[group_name])

    cv5_mean, cv5_std = evaluate_feature_set(selected_columns, cv_splits=5)
    refined_rows.append({
        "feature_set": row["feature_set"],
        "n_columns": row["n_columns"],
        "cv3_mean": row["cv3_mean"],
        "cv5_mean": cv5_mean,
        "cv5_std": cv5_std,
    })

top_feature_sets_df = pd.DataFrame(refined_rows).sort_values("cv5_mean", ascending=False)
print("\nTop 5 feature combinations by 5-fold CV f1_macro:")
print(top_feature_sets_df.head(5).to_string(index=False))

top_feature_sets_df.to_csv("model_feature_combo_results.csv", index=False)
print("\nSaved detailed results to model_feature_combo_results.csv")


Total combinations tested: 63
                                          feature_set  n_columns  cv3_mean
62  numeric_core + budget + location + holiday + g...         33  0.547091
60  numeric_core + location + holiday + genre_text...         29  0.545605
56  numeric_core + budget + location + holiday + g...         13  0.545110
58  numeric_core + budget + location + genre_text ...         32  0.545061
49  numeric_core + location + genre_text + genre_f...         28  0.544030
42      numeric_core + budget + location + genre_text         12  0.542993
47     numeric_core + location + holiday + genre_text          9  0.541848
26               numeric_core + location + genre_text          8  0.540203
57  numeric_core + budget + location + holiday + g...         32  0.537359
61  budget + location + holiday + genre_text + gen...         29  0.537244

Top 5 feature combinations by 5-fold CV f1_macro:
                                                          feature_set  n_columns  cv3_mean  cv

In [ ]:
# Using the best-ranked feature combination for downstream model search.
best_feature_set_name = top_feature_sets_df.iloc[0]["feature_set"]
best_feature_groups = best_feature_set_name.split(" + ")

best_columns = []
for group_name in best_feature_groups:
    best_columns.extend(feature_groups[group_name])

X_train = X_train_full[best_columns].copy()
X_test = X_test_full[best_columns].copy()

print("Best feature set:", best_feature_set_name)
print("Columns used:", len(best_columns))


Best feature set: numeric_core + budget + location + holiday + genre_text + genre_flags
Columns used: 33


In [7]:
# Model comparison on the best feature set.
preprocessor = make_preprocessor(X_train)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

searches = {
    "logistic_regression": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                solver="liblinear",
                random_state=SEED,
            )),
        ]),
        param_grid={"model__C": [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]},
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "random_forest": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", RandomForestClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced_subsample",
            )),
        ]),
        param_grid={
            "model__n_estimators": [200, 300, 500],
            "model__max_depth": [10, 30, 50],
            "model__min_samples_leaf": [1, 3, 5],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "extra_trees": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", ExtraTreesClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced",
            )),
        ]),
        param_grid={
            "model__n_estimators": [300, 400, 600],
            "model__max_depth": [10, 30, 50],
            "model__min_samples_leaf": [1, 3, 5],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
}

model_rows = []
best_name = None
best_search = None
best_cv_score = -np.inf

for name, search in searches.items():
    print(f"\n=== Training: {name} ===")
    search.fit(X_train, y_train)
    score = float(search.best_score_)
    model_rows.append({
        "model": name,
        "best_cv_f1_macro": score,
        "best_params": search.best_params_,
    })
    print(f"Best CV f1_macro ({name}): {score:.4f}")
    print(f"Best params ({name}): {search.best_params_}")

    if score > best_cv_score:
        best_cv_score = score
        best_name = name
        best_search = search

model_results_df = pd.DataFrame(model_rows).sort_values("best_cv_f1_macro", ascending=False)
print("\nModel ranking:")
print(model_results_df[["model", "best_cv_f1_macro"]])



=== Training: logistic_regression ===
Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best CV f1_macro (logistic_regression): 0.5495
Best params (logistic_regression): {'model__C': 0.05}

=== Training: random_forest ===
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best CV f1_macro (random_forest): 0.5862
Best params (random_forest): {'model__max_depth': 50, 'model__min_samples_leaf': 3, 'model__n_estimators': 500}

=== Training: extra_trees ===
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best CV f1_macro (extra_trees): 0.5792
Best params (extra_trees): {'model__max_depth': 30, 'model__min_samples_leaf': 1, 'model__n_estimators': 600}

Model ranking:
                 model  best_cv_f1_macro
1        random_forest          0.586232
2          extra_trees          0.579217
0  logistic_regression          0.549481


In [ ]:
best_model = best_search.best_estimator_

oof_proba = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)
f1_values = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = int(np.nanargmax(f1_values[:-1]))
best_threshold = float(thresholds[best_idx])

print("Selected model:", best_name)
print("Best threshold:", round(best_threshold, 4))
print("OOF F1 at best threshold:", round(float(f1_values[best_idx]), 4))


Selected model: random_forest
Best threshold: 0.2932
OOF F1 at best threshold: 0.8862


In [9]:
# Final holdout test estimate
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_f1_macro = f1_score(y_test, test_pred, average="macro")
test_f1_binary = f1_score(y_test, test_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Selected model:", best_name)
print("Selected feature set:", best_feature_set_name)
print("Estimated holdout test f1_macro:", round(float(test_f1_macro), 4))
print("Estimated holdout test f1 (positive class):", round(float(test_f1_binary), 4))
print("Estimated holdout test accuracy:", round(float(test_accuracy), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification report:")
print(classification_report(y_test, test_pred, digits=4))


Selected model: random_forest
Selected feature set: numeric_core + budget + location + holiday + genre_text + genre_flags
Estimated holdout test f1_macro: 0.4724
Estimated holdout test f1 (positive class): 0.8865
Estimated holdout test accuracy: 0.7974

Confusion matrix:
[[  158  5016]
 [   87 19924]]

Classification report:
              precision    recall  f1-score   support

           0     0.6449    0.0305    0.0583      5174
           1     0.7989    0.9957    0.8865     20011

    accuracy                         0.7974     25185
   macro avg     0.7219    0.5131    0.4724     25185
weighted avg     0.7672    0.7974    0.7163     25185

